In [1]:
import os
import json
import time
import re
import pandas as pd
import concurrent.futures
import httpx
from tqdm import tqdm
from openai import OpenAI

In [ ]:
TOGETHER_API_KEY = "ĐIỀN_API_KEY_TOGETHER_CỦA_BẠN_VÀO_ĐÂY"

# Khởi tạo Client trỏ về server của Together
client = OpenAI(
    api_key=TOGETHER_API_KEY,
    base_url="https://api.together.xyz/v1",
    timeout=httpx.Timeout(120.0)
)

# Chạy Phase 1: Tạo Commonsense bằng Llama 70B
MODEL_COMMONSENSE = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

In [3]:
def get_together_batch_commonsense(batch_data):
    system_prompt = """Bạn là một TỪ ĐIỂN VĂN HÓA MẠNG VÀ BÁCH KHOA TOÀN THƯ trung lập.

Nhiệm vụ của bạn là cung cấp kiến thức nền (Commonsense) để một người nước ngoài có thể hiểu được bình luận, dựa trên ngữ cảnh video.

LUẬT THÉP BẮT BUỘC ĐỂ TRÁNH RÒ RỈ DỮ LIỆU (DATA LEAKAGE) VÀ LỖI TRÔI NHÃN:
1. BẮT BUỘC TRUNG LẬP: Chỉ giải nghĩa từ lóng, sửa lỗi chính tả và cung cấp sự thật khách quan (Fact).
2. CẤM SỬ DỤNG TỪ KHÓA CẢM XÚC/THÁI ĐỘ: TUYỆT ĐỐI KHÔNG xuất hiện các từ: "mỉa mai", "châm biếm", "hài hước", "đá đểu", "giễu nhại", "khen ngợi", "ngạc nhiên", "sốc", "chê trách".
3. CẤM PHÂN TÍCH Ý ĐỒ: Không được đoán xem người dùng đang nghĩ gì hay muốn làm gì. Đừng viết "Người dùng có ý nói..." hay "Câu này nhằm mục đích...".
4. CẤU TRÚC 2 CÂU CHUẨN:
- Câu 1: Giải nghĩa từ lóng, teencode, phiên âm, hoặc lỗi gõ phím trong bối cảnh của bình luận và ngữ cảnh video.
- Câu 2: Cung cấp Fact (sự thật lịch sử, đặc điểm nhân vật, hoặc chi tiết video) liên quan đến bình luận.
5. TRẢ VỀ JSON hợp lệ chứa mảng "results", value của "commonsense" phải là STRING.
6. Nếu trích dẫn bình luận thì phải trích dẫn chính xác đoạn có nghĩa, không cố tự bịa ra nghĩa của trích dẫn, không được cắt ngang từ để tránh hiểu sai.
7. Không đưa ra lời khuyên và chữa lỗi cách dùng từ của người dùng, chỉ giải thích nghĩa của từ/cụm từ đó trong ngữ cảnh bình luận và video.
8. KHÔNG ĐƯỢC NHẦM ID: Trong JSON trả về, BẮT BUỘC phải có trường "comment" trích dẫn lại nguyên văn bình luận tương ứng với "id" đó để làm mỏ neo, tránh gán nhầm đáp án của câu này sang câu khác.


=== VÍ DỤ CHUẨN (THUẦN THÔNG TIN, KHÔNG CẢM XÚC) ===

[Input]
ID: 101 | Context: Video về Pablo Picasso. | Comment: oát đờ cái tên khai sinh
ID: 102 | Context: Video về Pablo Picasso. | Comment: cấm sừng nhiều thế
ID: 103 | Context: Video về Pablo Picasso. | Comment: bác pi cát xô là bạn của bác hồ
ID: 104 | Context: Video vẽ tranh lập thể của Picasso. | Comment: những người đó đều là nicki minaj nhé
ID: 105 | Context: Video trận bóng đá. | Comment: blv đọc văn tế à

[Output (JSON)]
{
  "results": [
    {"id": 101, "comment": "oát đờ cái tên khai sinh", "commonsense": "Từ 'oát đờ' là cách đọc phiên âm tiếng Việt của cụm từ tiếng Anh 'What the...'. Tên khai sinh đầy đủ của họa sĩ Picasso bao gồm 23 từ ghép lại với nhau."},
    {"id": 102, "comment": "cấm sừng nhiều thế", "commonsense": "'Cấm sừng' là lỗi gõ phím của từ 'cắm sừng', một tiếng lóng mạng chỉ hành vi ngoại tình. Trong thực tế, Picasso được biết đến là người có rất nhiều người tình và các mối quan hệ ngoài luồng."},
    {"id": 103, "comment": "bác pi cát xô là bạn của bác hồ", "commonsense": "Theo một số ghi chép lịch sử và giai thoại, Chủ tịch Hồ Chí Minh và danh họa Picasso từng có dịp gặp gỡ và giao lưu với nhau trong thời gian ở Pháp."},
    {"id": 104, "comment": "những người đó đều là nicki minaj nhé", "commonsense": "Trường phái hội họa Lập thể của Picasso thường vẽ phụ nữ với các khối hình học và tỷ lệ cơ thể biến dạng. Nicki Minaj là một nữ ca sĩ người Mỹ nổi tiếng với vóc dáng phẫu thuật thẩm mỹ có tỷ lệ ba vòng cường điệu."},
    {"id": 105, "comment": "blv đọc văn tế à", "commonsense": "Cụm từ 'đọc văn tế' chỉ việc đọc một bài văn khóc người chết với âm điệu đều đều, buồn bã và thê lương. Bình luận viên bóng đá là người chịu trách nhiệm truyền tải không khí sôi động của trận đấu đến khán giả."}
  ]
}
=============================="""

    # Sắp xếp lại User Prompt nằm trên 1 dòng để giống hệt ví dụ mẫu của bạn
    user_prompt = "[Input]\n"
    for item in batch_data:
        user_prompt += f"ID: {item['id']} | Context: {item['ctx']} | Comment: {item['cmt']}\n"
    user_prompt += "\n[Output (JSON)]:"

    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL_COMMONSENSE,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.0,  # Đưa về 0.0 để loại bỏ tính ngẫu nhiên gây lệch ID
                response_format={"type": "json_object"}
            )
            
            content = response.choices[0].message.content
            if not content: return None
                
            try:
                content = re.sub(r'[\x00-\x1f]', '', content)
                result_json = json.loads(content, strict=False)
                return result_json.get("results", [])
            except json.JSONDecodeError:
                return None

        except Exception as e:
            from tqdm import tqdm
            import time
            tqdm.write(f"⚠️ Lỗi API (Lần {attempt+1}/3): {str(e)}")
            time.sleep(3)
            
    return None

In [4]:
def process_single_batch(batch_info):
    """Hàm bọc để chạy ngầm trên một luồng riêng biệt"""
    batch_idx, batch_data = batch_info
    results = get_together_batch_commonsense(batch_data)
    return batch_idx, results

In [5]:
def process_dataset_multithread(input_csv, output_csv, batch_size=15, max_workers=3):
    if os.path.exists(output_csv):
        print(f"\n[*] Tìm thấy file đang chạy dở: {output_csv}. Đang resume...")
        df = pd.read_csv(output_csv)
    else:
        print(f"\n[*] Chạy lần đầu. Đang đọc file gốc: {input_csv}")
        df = pd.read_csv(input_csv)
        
        print("[*] Đang TIÊU DIỆT toàn bộ dữ liệu cột 'commonsense' cũ...")
        df['commonsense'] = None 
        
        df.to_csv(output_csv, index=False, encoding='utf-8-sig')
        print(f"[*] Đã tạo file giấy trắng chuẩn bị chạy Batch: {output_csv}")
    
    if 'commonsense' not in df.columns:
        df['commonsense'] = None

    def needs_processing(x):
        val = str(x).strip()
        if val in ['nan', 'None', '']: return True
        if val.startswith("Lỗi"): return True
        if "[" in val or "]" in val: return True
        return False

    pending_indices = df[df['commonsense'].apply(needs_processing)].index.tolist()
    print(f"Tổng số dòng đang chờ Llama 70B: {len(pending_indices)} | Batch Size: {batch_size} | Luồng: {max_workers}")
    
    if not pending_indices:
        print("Mọi dữ liệu đã được làm xong 100%!")
        return

    # 4.1 Chia nhỏ dữ liệu thành các cục (Batches)
    all_batches = []
    for i in range(0, len(pending_indices), batch_size):
        batch_indices = pending_indices[i:i+batch_size]
        batch_data = []
        for idx in batch_indices:
            ctx = str(df.at[idx, 'video_core_content']).replace('\n', ' ')
            cmt = str(df.at[idx, 'comment']).replace('\n', ' ')
            if ctx.lower() == 'nan': ctx = ''
            if cmt.lower() == 'nan': cmt = ''
            batch_data.append({"id": idx, "ctx": ctx, "cmt": cmt})
        
        all_batches.append((i // batch_size, batch_data))

    print(f"🚀 Phóng {max_workers} luồng tên lửa oanh tạc API...")

    # 4.2 Kích hoạt Đa luồng (Multi-threading)
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_single_batch, b) for b in all_batches]
        
        # as_completed: Thằng nào xong trước thì báo cáo trước
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Đang cày (Llama 70B)"):
            batch_idx, results = future.result()
            
            if results:
                # Ghi kết quả vào DataFrame (Ghi ở Main Thread nên rất an toàn)
                for res in results:
                    row_id = res.get("id")
                    ans = res.get("commonsense", "")
                    
                    if isinstance(ans, list): 
                        ans = " ".join([str(item) for item in ans])
                        
                    if row_id is not None and row_id in df.index:
                        df.at[row_id, 'commonsense'] = str(ans).strip()
            
            # Ghi đè file CSV ngay sau khi mỗi Batch cập bến thành công
            df.to_csv(output_csv, index=False, encoding='utf-8-sig')

    print(f"\n[HOÀN THÀNH PHASE 1] Dữ liệu Commonsense siêu chất lượng đã lưu tại: {output_csv}")

In [8]:
if __name__ == "__main__":
    INPUT_FILE = "combined_data.csv" 
    OUTPUT_FILE = "silver_70B.csv" 
    
    # === THÔNG SỐ TỐI ƯU CHO LLAMA 70B ===
    BATCH_SIZE = 15    # Đủ lớn để tiết kiệm token, đủ nhỏ để không bị lú
    MAX_WORKERS = 5    # Mở 5 luồng bắn API cùng lúc. Nếu mượt, bạn có thể thử tăng lên 5!
    
    if os.path.exists(INPUT_FILE):
        process_dataset_multithread(INPUT_FILE, OUTPUT_FILE, batch_size=BATCH_SIZE, max_workers=MAX_WORKERS)
    else:
        print(f"Lỗi: Không tìm thấy file gốc {INPUT_FILE}")


[*] Tìm thấy file đang chạy dở: silver_70B.csv. Đang resume...
Tổng số dòng đang chờ Llama 70B: 1431 | Batch Size: 15 | Luồng: 5
🚀 Phóng 5 luồng tên lửa oanh tạc API...


Đang cày (Llama 70B): 100%|██████████| 96/96 [15:06<00:00,  9.45s/it]


[HOÀN THÀNH PHASE 1] Dữ liệu Commonsense siêu chất lượng đã lưu tại: silver_70B.csv


In [7]:
import pandas as pd
import re

file_path = "silver_70B.csv"
print(f"⏳ Đang khởi động Radar quét lỗi Sequence Shift diện rộng trên {file_path}...")
df = pd.read_csv(file_path)

shifted_count = 0

for i in range(len(df)):
    ans = str(df.at[i, 'commonsense']).lower()
    
    # Bỏ qua những dòng chưa làm hoặc trống
    if ans in ['nan', 'none', '']:
        continue
        
    current_cmt = str(df.at[i, 'comment']).strip().lower()
    is_shifted = False
    
    # ==========================================
    # LƯỚI LỌC 1: BẮT MỌI KIỂU TRÍCH DẪN ĐẦU CÂU
    # ==========================================
    # Bắt các pattern: "Từ '...' ", "Cụm từ '...' ", "Câu '...' ", hoặc chỉ mở đầu bằng "'...'"
    matches = re.findall(r"(?:từ|cụm từ|câu|chữ)?\s*['\"]([^'\"]+)['\"]", ans)
    
    for quoted_text in matches:
        quoted_text = quoted_text.strip()
        # Bỏ qua các từ quá ngắn như 'à', 'ừ'
        if len(quoted_text) > 2:
            # Tách thành các từ đơn để so sánh (tránh việc LLM tự sửa lỗi chính tả khiến string ko khớp 100%)
            quoted_words = set(quoted_text.split())
            cmt_words = set(current_cmt.split())
            
            # Đếm số từ trùng lặp giữa cái nó trích dẫn và bình luận gốc
            overlap = quoted_words.intersection(cmt_words)
            
            # Nếu tỷ lệ trùng khớp dưới 40% -> Rõ ràng nó đang nói về câu của người khác!
            if len(quoted_words) > 0 and len(overlap) / len(quoted_words) < 0.4:
                is_shifted = True
                break

    # ==========================================
    # LƯỚI LỌC 2: QUÉT HÀNG XÓM BÁN KÍNH 4 DÒNG
    # ==========================================
    if not is_shifted:
        neighbors = []
        # Quét 2 dòng trên và 2 dòng dưới
        for step in [-2, -1, 1, 2]:
            if 0 <= i + step < len(df):
                neighbor_cmt = str(df.at[i+step, 'comment']).strip().lower()
                # Chỉ lấy hàng xóm có comment dài > 4 ký tự để làm mốc so sánh
                if len(neighbor_cmt) > 4: 
                    neighbors.append(neighbor_cmt)
                    
        for neighbor in neighbors:
            # Nếu câu trả lời chứa nguyên văn câu của ông hàng xóm, mà lại ko chứa câu của chính nó
            if neighbor in ans and current_cmt not in ans:
                is_shifted = True
                break
                
    # Nếu dính lưới, tử hình (xóa trắng)
    if is_shifted:
        df.at[i, 'commonsense'] = None
        shifted_count += 1

print(f"🔥 Radar đã phát hiện và XÓA TRẮNG thêm {shifted_count} dòng bị lệch nhãn ẩn sâu.")

# Lưu đè file an toàn
df.to_csv(file_path, index=False, encoding='utf-8-sig')
print("✅ Tập dữ liệu đã được thanh tẩy thành công!")

⏳ Đang khởi động Radar quét lỗi Sequence Shift diện rộng trên silver_70B.csv...
🔥 Radar đã phát hiện và XÓA TRẮNG thêm 1227 dòng bị lệch nhãn ẩn sâu.
✅ Tập dữ liệu đã được thanh tẩy thành công!
